# 02 — Reanálisis corregido de los modelos
**Paper Cañete · Royal Society Open Science**

Tres evaluaciones sobre la misma grilla, solo con **variables de paisaje**
(`altitud, pendiente, dist_acequia, dist_rio`):
- **B.** Partición aleatoria estratificada → SMOTE solo en train.
- **C.** Validación cruzada espacial por bloques de 2 km (GroupKFold, 5 folds) — la evaluación honesta.
- **Baselines**: prevalencia y heurística no-ML ("alertar cerca de acequia").

⚠️ Tiempos: B ~1–2 min; C ~3–5 min (con checkpoint por fold, es resumible).
Para las cifras finales del paper subir `N_ESTIMATORS=100` y quitar `MAX_DEPTH`.

> ⚠️ **Si aparece `ModuleNotFoundError: matplotlib.backends.registry`**: es una mezcla de versiones
> de matplotlib (necesita ≥3.9). Ejecuta la celda de setup de abajo y **reinicia el kernel/runtime**
> (en Colab: Entorno de ejecución → Reiniciar sesión). El error desaparece tras el reinicio.

In [ ]:
# ── Celda de setup: ejecutar UNA vez; si instala algo, REINICIAR el kernel/runtime ──
import importlib, subprocess, sys
def asegurar(pkg, spec):
    try:
        m = importlib.import_module(pkg)
        if pkg == 'matplotlib':
            v = tuple(int(x) for x in m.__version__.split('.')[:2])
            if v < (3, 9): raise ImportError
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', spec])
        print(f"instalado/actualizado: {spec}  ->  REINICIA el kernel y vuelve a ejecutar")
for pkg, spec in [('matplotlib','matplotlib>=3.9'), ('imblearn','imbalanced-learn'),
                  ('sklearn','scikit-learn'), ('pandas','pandas'), ('scipy','scipy')]:
    asegurar(pkg, spec)
import matplotlib
print(f"matplotlib {matplotlib.__version__} OK — entorno listo")

In [ ]:
import pandas as pd, numpy as np, warnings, os, json
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score)
from imblearn.over_sampling import SMOTE

# --- Parámetros (ligeros para exploración; ver nota arriba para el paper) ---
N_ESTIMATORS = 30
MAX_DEPTH    = 12      # None para el paper (requiere ~8+ GB de RAM)
BLOCK_M      = 2000    # tamaño del bloque espacial; sensibilidad: 1000 y 4000
PAISAJE = ['altitud','pendiente','dist_acequia','dist_rio']

df = pd.read_csv("../datos/grilla_modelo2_620k.csv.gz", low_memory=False)
df = df.dropna(subset=PAISAJE).reset_index(drop=True)
y = df['label'].values
print(f"Celdas: {len(df):,} | positivas: {int(y.sum())} | prevalencia: {y.mean():.5%}")

def reporte(nombre, y_true, proba, umbral=0.5):
    pred = (proba >= umbral).astype(int)
    r = dict(pipeline=nombre,
             accuracy=round(accuracy_score(y_true,pred),4),
             precision=round(precision_score(y_true,pred,zero_division=0),4),
             recall=round(recall_score(y_true,pred,zero_division=0),4),
             f1=round(f1_score(y_true,pred,zero_division=0),4),
             pr_auc=round(average_precision_score(y_true,proba),5))
    print(r); return r

## B — Partición aleatoria corregida (SMOTE solo en train)

In [ ]:
X = df[PAISAJE].values.astype(np.float32)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
Xr, yr = SMOTE(random_state=42).fit_resample(Xtr, ytr)
m = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                           random_state=42, n_jobs=-1, max_samples=0.5).fit(Xr, yr)
res_B = reporte('B_aleatoria_SMOTE_en_train', yte, m.predict_proba(Xte)[:,1])

## C — Validación cruzada espacial por bloques (la evaluación honesta)
Cada fold entrena en unos bloques de 2×2 km y predice en bloques **nunca vistos**,
rompiendo la autocorrelación espacial. Con checkpoint por fold.

In [ ]:
df['block'] = (df['x']//BLOCK_M).astype(int).astype(str)+'_'+(df['y']//BLOCK_M).astype(int).astype(str)
g = df['block'].values
print(f"bloques: {df['block'].nunique()}")
pp = np.zeros(len(y))
for k,(tr, te) in enumerate(GroupKFold(n_splits=5).split(X, y, g)):
    ckpt = f"_fold{k}.npz"
    if os.path.exists(ckpt):
        d = np.load(ckpt); pp[d['te']] = d['proba']; print(f"fold {k}: checkpoint"); continue
    npos = int(y[tr].sum())
    Xr, yr = SMOTE(random_state=42, k_neighbors=min(5, npos-1)).fit_resample(X[tr], y[tr])
    m = RandomForestClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
                               random_state=42, n_jobs=-1, max_samples=0.5).fit(Xr, yr)
    pp[te] = m.predict_proba(X[te])[:,1]
    np.savez(ckpt, te=te, proba=pp[te]); print(f"fold {k}: ok (train_pos={npos})")
res_C = reporte('C_spatial_block_CV', y, pp)

## Precisión@K y enriquecimiento — la métrica del *triage*
El presupuesto de campo real fue ~146 alertas (Modelo 2). La pregunta correcta no es
"¿clasifica bien cada celda?" sino "¿cuánto enriquece las K celdas priorizadas?".

In [ ]:
orden = np.argsort(-pp)
prev = y.mean()
print(f"prevalencia base: {prev:.5%}")
for K in (50, 100, 146, 500):
    pk = y[orden[:K]].mean()
    print(f"precision@{K:>3}: {pk:.4f}  | enriquecimiento: {pk/prev:6.1f}x")

## Baselines no-ML (obligatorios para el paper)

In [ ]:
prev = y.mean()
# Baseline 1: clasificador de prevalencia (siempre 0)
print(f"Baseline prevalencia — accuracy: {(y==0).mean():.5f}, recall: 0, PR-AUC≈{y.mean():.5f}")

# Baseline 2: heurística 'alertar toda celda a < X m de acequia'
for X_m in (50, 100, 200):
    pred_h = (df['dist_acequia'] < X_m).astype(int).values
    pr = precision_score(y, pred_h, zero_division=0); rc = recall_score(y, pred_h)
    print(f"heurística acequia<{X_m:>3}m — precisión: {pr:.4f} ({pr/prev:.0f}x), recall: {rc:.3f}, alertas: {pred_h.sum():,}")

In [ ]:
# Exportar predicciones y resultados
df[['x','y','label']].assign(proba_spatial_cv=pp).to_csv('predicciones_spatial_cv.csv', index=False)
json.dump([res_B, res_C], open('resultados_reanalisis.json','w'), indent=1)
print('Guardado: predicciones_spatial_cv.csv, resultados_reanalisis.json')

## Lectura de resultados
- **PR-AUC honesto ~0.003** (prevalencia 0.0003): a nivel de celda de 50 m, con 4 variables,
  el problema es durísimo — coherente con la arqueología.
- Pero el ranking **enriquece ~20×** sobre la prevalencia en el presupuesto de 146 alertas,
  y la heurística de acequias muestra cuánto de eso es hidrología pura (comparación clave del paper).
- La validación definitiva del ranking no está aquí: está en el campo (→ notebook 03).